# TenderWin · Лід і мейл генератор v1.1 — ТЕСТОВИЙ РЕЖИМ

Сім клітинок, згори вниз. Код редагувати не треба.

**Що робить:** шукає в Prozorro відхилення за поточну добу по всіх галузях і
всіх закупівлях, складає перший лист і робочу картку закупівлі.

**Чого не робить:** не завантажує документи, не аналізує їх, не пише скарг і
**не надсилає нічого клієнтам** — тільки на ваші тестові скриньки.

Потрібні три файли: `tenderwin_lead_mail.py`, `tenderwin_lead_engine.py`,
`lead_machine_v1.py`. Якщо їх ще немає на Диску — клітинка 2 запропонує
завантажити їх із компʼютера.

## КЛІТИНКА 1 · Встановлення бібліотек

In [1]:
!pip -q install google-api-python-client google-auth google-auth-oauthlib python-docx requests
print("Готово. Далі — клітинка 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 1.7 MB/s eta 0:00:00
Готово. Далі — клітинка 2.


## КЛІТИНКА 2 · Google Диск і чотири файли

Монтує Диск, знаходить теку `TenderWin` (будь-який регістр) і перевіряє файли.
Якщо якогось бракує — одразу відкриється вікно вибору файлів на компʼютері.

Потрібні: `tenderwin_lead_mail.py`, `tenderwin_lead_engine.py`,
`lead_machine_v1.py`, `tenderwin_service_cards.py` (службові картки).

In [2]:
from google.colab import drive
import os, re, shutil

drive.mount('/content/drive')

КОРІНЬ = '/content/drive/MyDrive'
ІМЕНА_ТЕКИ = ('tenderwin',)          # будь-який регістр: TENDERWIN, TenderWin…
ПОТРІБНІ = ['tenderwin_lead_mail.py', 'tenderwin_lead_engine.py',
            'lead_machine_v1.py', 'tenderwin_service_cards.py']
ОЗНАКИ = ['.gmail_token.json', 'leads.db', 'tenderwin_lead_mail.db']


def вміст(тека):
    try:
        return set(os.listdir(тека))
    except OSError:
        return set()


def вага(тека):
    """Що більше в теці нашого — то вона і є справжня."""
    файли = вміст(тека)
    return (sum(1 for н in ПОТРІБНІ if н in файли),
            sum(1 for н in ОЗНАКИ if н in файли))


кандидати = sorted(
    (os.path.join(КОРІНЬ, назва) for назва in os.listdir(КОРІНЬ)
     if назва.lower() in ІМЕНА_ТЕКИ and os.path.isdir(os.path.join(КОРІНЬ, назва))),
    key=вага, reverse=True)

if кандидати:
    DYSK = кандидати[0]
    if len(кандидати) > 1:
        print("УВАГА: на Диску кілька тек TenderWin:")
        for тека in кандидати:
            є_файлів, є_ознак = вага(тека)
            print(f"   {тека}  (наших файлів: {є_файлів}, бази й токен: {є_ознак})")
        print("   Працюю з першою. Зайву можна видалити руками — але спершу")
        print("   перевірте, що в ній нічого потрібного не лишилось.")
else:
    DYSK = os.path.join(КОРІНЬ, 'TENDERWIN')
    os.makedirs(DYSK, exist_ok=True)
    print("Теки TenderWin не було — створив:", DYSK)

print("Робоча тека:", DYSK)


def чого_бракує():
    файли = вміст(DYSK)
    return [назва for назва in ПОТРІБНІ if назва not in файли]


def справжня_назва(назва):
    # Браузер часто додає "(1)": tenderwin_lead_mail (1).py
    чиста = re.sub(r"\s*\(\d+\)(?=\.py$)", "", назва).strip()
    return чиста if чиста in ПОТРІБНІ else назва


бракує = чого_бракує()
ГОТОВО = not бракує

if ГОТОВО:
    print("Усі три файли на місці. Далі — клітинка 3.")
else:
    print("Бракує:", ", ".join(бракує))
    print("Оберіть ці файли на компʼютері у вікні нижче.")
    print("(Або покладіть їх у цю теку на Диску і виконайте клітинку ще раз.)")
    try:
        from google.colab import files
        завантажені = files.upload()
        for тимчасова in list(завантажені):
            назва = справжня_назва(тимчасова)
            if назва in ПОТРІБНІ:
                куди = os.path.join(DYSK, назва)
                if os.path.exists(куди):
                    os.remove(куди)
                shutil.move(тимчасова, куди)
                print("  покладено на Диск:", назва)
            else:
                print("  пропущено (не потрібен тут):", тимчасова)
    except Exception as помилка:
        print("Вікно вибору не відкрилося:", type(помилка).__name__)

    бракує = чого_бракує()
    ГОТОВО = not бракує
    print()
    print("Усі три файли на місці. Далі — клітинка 3." if ГОТОВО
          else "Досі бракує: " + ", ".join(бракує) + ". Виконайте клітинку ще раз.")

Mounted at /content/drive
УВАГА: на Диску кілька тек TenderWin:
   /content/drive/MyDrive/TENDERWIN  (наших файлів: 3, бази й токен: 3)
   /content/drive/MyDrive/TenderWin  (наших файлів: 3, бази й токен: 1)
   Працюю з першою. Зайву можна видалити руками — але спершу
   перевірте, що в ній нічого потрібного не лишилось.
Робоча тека: /content/drive/MyDrive/TENDERWIN
Усі три файли на місці. Далі — клітинка 3.


## КЛІТИНКА 3 · Доступ до пошти

Секрет `GMAIL_OAUTH_JSON` має бути в Colab Secrets (ключик ліворуч), доступ
для цього ноутбука — увімкнений. У код секрети не вставляються ніколи.

In [3]:
import sys, os

if "DYSK" not in globals():
    print("Спершу виконайте клітинку 2 — вона монтує Диск.")
    ФАЙЛИ_Є = False
else:
    if DYSK not in sys.path:
        sys.path.insert(0, DYSK)
    try:
        import tenderwin_lead_engine as E
        ФАЙЛИ_Є = True
    except ModuleNotFoundError as помилка:
        ФАЙЛИ_Є = False
        print("Немає файла:", (помилка.name or "?") + ".py")
        print("Поверніться до клітинки 2 — вона покладе файли на Диск.")

if ФАЙЛИ_Є:
    # Без цього рядка токен, що лежить файлом на Диску, просто не видно:
    # движок шукає .gmail_token.json у теці, яку йому назвали.
    E.TOKEN_DIR = DYSK
    у_теці = os.path.exists(os.path.join(DYSK, E.TOKEN_FILE_NAME))

    if E._get_secret(E.GMAIL_OAUTH_ENV):
        print("Доступ до пошти є" + (f" (файл {E.TOKEN_FILE_NAME} у теці на Диску)."
                                     if у_теці else " (секрет Colab)."))
        бракує_дозволів = E.missing_scopes()
        if бракує_дозволів:
            print("  Але токену бракує дозволів:", ", ".join(бракує_дозволів))
            print("  Зробіть E.reset_gmail_token(DYSK), потім E.mint_oauth_token().")
        else:
            print("Далі — клітинка 4.")
        if у_теці:
            print("  ⓘ Токен у теці на Диску може прочитати кожен, хто має")
            print("    доступ до цієї теки. Надійніше тримати його в Colab Secrets.")
    else:
        print(f"Доступу до пошти немає: ні секрету {E.GMAIL_OAUTH_ENV},")
        print(f"ні файла {E.TOKEN_FILE_NAME} у теці {DYSK}.")
        print("Без нього працює сухий прогін (клітинка 6), відправки не буде.")
        print("Як отримати токен: E.mint_oauth_token()")

Доступ до пошти є (файл .gmail_token.json у теці на Диску).
Далі — клітинка 4.
  ⓘ Токен у теці на Диску може прочитати кожен, хто має
    доступ до цієї теки. Надійніше тримати його в Colab Secrets.


## КЛІТИНКА 4 · Завантаження скрипта

In [4]:
import importlib

M = None
if "DYSK" not in globals():
    print("Спершу виконайте клітинку 2.")
else:
    try:
        import tenderwin_lead_mail as M
        importlib.reload(M)
    except ModuleNotFoundError as помилка:
        M = None
        print("Немає файла:", (помилка.name or "?") + ".py")
        print("Поверніться до клітинки 2.")

if M is not None:
    M.setup(DYSK)      # база, білий список адрес, історія попереднього движка

  TENDERWIN LEAD & MAIL 1.2.0 від 2026-09-23 · підготовка
  режим TEST · шаблон листа V8_OWNER
  база: /content/drive/MyDrive/TENDERWIN/tenderwin_lead_mail.db
  вихід: /content/drive/MyDrive/TENDERWIN/vyhid_lystiv
  тестових адрес у білому списку: 16
  історія попереднього движка: вже перенесена раніше

  Далі: M.go() — сухий прогін за сьогодні.


## КЛІТИНКА 5 · Налаштування

Змінювати не обовʼязково. Значення нижче — чинне рішення власника.

In [5]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.LETTER_VERSION = "V8_OWNER"       # ЧИННИЙ лист (рішення 23.09.2026)
    M.LETTER_FORMAT  = "HTML_AND_TEXT"  # "TEXT_ONLY" — лист лише текстом
    M.WINDOW_MODE    = "TODAY_ONLY"     # лише поточна доба
    M.SKIP_WITHOUT_COMPLAINT_ROUTE = True   # без строку оскарження — не пишемо
    M.MIN_LOT_VALUE  = 0                # 0 = без порогу вартості
    M.SERVICE_CARDS  = True             # службові картки з протоколами
    M.SERVICE_CARDS_FRESH = True        # свіжий стан рішення з Prozorro

    print("режим:", M.MODE, "· лист:", M.LETTER_VERSION, "·", M.LETTER_FORMAT)
    print("адрес для тестової доставки:", len(M.TEST_RECIPIENTS))
    print("службові картки:", "так" if M.SERVICE_CARDS else "ні")

режим: TEST · лист: V6_OWNER · HTML_AND_TEXT
адрес для тестової доставки: 16


## КЛІТИНКА 6 · Сухий прогін

Шукає відхилення за сьогодні, складає листи у файли і робочі картки.
**Нічого нікуди не надсилає.**

У теці `vyhid_lystiv/eml` біля кожного листа лежить файл `.html` — відкрийте
його у браузері, щоб побачити лист таким, яким його побачить клієнт.

In [6]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    результат = M.go(dysk=DYSK)

  TENDERWIN LEAD & MAIL 1.2.0 від 2026-09-23 · сухий прогін
  режим TEST · шаблон листа V6_OWNER

КРОК 1/3. Читаю стрічку змін Prozorro (02.10.2026 00:00 — 02.10.2026 19:06 (поточна доба))…
    подій у стрічці: 19,564 • закупівель до перевірки: 3,215
    пропущено за типом процедури: 13,215

КРОК 2/3. Перевіряю закупівлі (12 потоків)…
    перевірено 500/3215 • подій у вікні: 0
    перевірено 1000/3215 • подій у вікні: 28
    перевірено 1500/3215 • подій у вікні: 53
    перевірено 2000/3215 • подій у вікні: 75
    перевірено 2500/3215 • подій у вікні: 101
    перевірено 3000/3215 • подій у вікні: 128


    перевірено 3215 • подій у вікні: 144 • на перевірку: 0

------------------------------------------------------------------------------
  ПІДСУМОК ПРОГОНУ R20261002-190632-c26d
------------------------------------------------------------------------------
  Вікно: 02.10.2026 00:00 — 02.10.2026 19:06 (поточна доба)
  Обхід стрічки: COMPLETE
  Закупівель перевірено: 3 215
  Пропущено за типом процедури: 13 215
     reporting: 9 576 — звіт про договір без використання системи
     priceQuotation: 2 791 — запит ціни пропозицій
     belowThreshold: 776 — допорогова (спрощена) закупівля
     requestForProposal: 72 — не в переліку дозволених
     ⓘ Рішення власника: відхилення в цих процедурах не оскаржуються в АМКУ, тому листів по них немає.
  Відхилень у вікні: 144 (усього побачено 423)
     старіші за вікно: 272
     ! з’явилися після попереднього прогону, але вчора: 7 — за нинішнім налаштуванням WINDOW_MODE='TODAY_ONLY' вони не потраплять у розсилку ніколи
     ненадійний час події: 0 

## КЛІТИНКА 7 · Відправка на ваші скриньки і стан бази

Листи підуть **тільки** на адреси зі списку `TEST_RECIPIENTS`. База це
перевіряє окремо від коду: адресу клієнта туди не пропустить тригер.

In [7]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.send(dysk=DYSK)     # відправка на тестові скриньки + звірка з Gmail
    M.state(dysk=DYSK)    # що зараз у базі

  TENDERWIN LEAD & MAIL 1.2.0 від 2026-09-23 · відправка на тестові скриньки
  режим TEST · шаблон листа V6_OWNER
    [1/25] SENT_CONFIRMED → ppvetik4@gmail.com
    пауза 174 с …
    [2/25] SENT_CONFIRMED → rebuilduaprojects@gmail.com
    пауза 131 с …
    [3/25] SENT_CONFIRMED → ppvetik.ecomerce@gmail.com
    пауза 167 с …
    [4/25] SENT_CONFIRMED → vitaliishchaslyvyi@gmail.com
    пауза 158 с …
    [5/25] SENT_CONFIRMED → kiev.development.company@gmail.com
    пауза 165 с …
    [6/25] SENT_CONFIRMED → mike.vital.llc@gmail.com
    пауза 168 с …
    [7/25] SENT_CONFIRMED → t.shchaslyva@gmail.com
    пауза 127 с …
    [8/25] SENT_CONFIRMED → saveukrainekdk@gmail.com
    пауза 144 с …
    [9/25] SENT_CONFIRMED → saveukrainekdk@gmail.com
    пауза 171 с …
    [10/25] SENT_CONFIRMED → vtgglibalsolutions@gmail.com
    пауза 138 с …
    [11/25] SENT_CONFIRMED → mike.vital.llc@gmail.com
    пауза 150 с …
    [12/25] SENT_CONFIRMED → mike.vital.llc@gmail.com
    пауза 154 с …
    [13/25] SENT

## КЛІТИНКА 8 · Службові картки ще раз (за потреби)

Після `M.go()` картки вже зроблені. Ця клітинка потрібна лише тоді, коли
крок карток обірвався або коли треба зібрати картки за інший день.

Уже завантажені файли вдруге не качаються. Картку, у якій ви щось дописали,
скрипт **не перезаписує** — нова версія ляже поруч окремим файлом.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.kartky(dysk=DYSK)                      # події останнього прогону
    # M.kartky(day="2026-10-02", dysk=DYSK)  # усі події конкретного дня

---
### Якщо щось пішло не так

| Повідомлення | Що робити |
|---|---|
| `Немає файла: …py` | клітинка 2 — покладіть файли на Диск |
| `TRANSPORT_UNAVAILABLE` | немає секрету `GMAIL_OAUTH_JSON` — клітинка 3 |
| `TEST_ALLOWLIST_EMPTY` | порожній `TEST_RECIPIENTS` — так задумано, заповніть |
| `PROZORRO_UNAVAILABLE` | Prozorro не відповідає — спробуйте за кілька хвилин |
| `Обхід стрічки: INCOMPLETE` | прочитано не всю стрічку; у звіті написано чому |
| `листи без певного стану` | зробіть `M.send(dysk=DYSK)` — він звірить їх із Gmail |
| `SERVICE_CARDS_MODULE_MISSING` | немає `tenderwin_service_cards.py` — клітинка 2; листи цим не зачеплені |
| `SERVICE_CARDS_FAILED` | крок карток обірвався; листи збережено. Повторіть `M.kartky(dysk=DYSK)` |

**Де службові картки:** `TenderWin → Службові картки → <дата> → <UA-ID>__<код>__<id>`.
У теці дня лежить `_ПЕРЕЛІК_<прогін>.xlsx` — з нього відбирають справи.

Перевірити сам скрипт: `M.tests(DYSK)`